# §0 Setup (Stage S0)

This section fixes the foundations every later result depends on: the software environment, the run mode, the random-number streams, the integrity of the raw data, and the repository layout. A reader (or marker) who re-runs the notebook should obtain the same numbers; the checks below make a failure visible at the start rather than as a silent difference at the end.

| Check | Why it exists |
|---|---|
| Environment table | Library versions change results (solver defaults, spline code). Recording them makes a deviation explainable. |
| Run mode | FAST lets a grader re-run in minutes; FINAL is the protocol whose outputs are stored in the submitted notebook. |
| Seed derivation | One master seed, independent child streams: re-running a block never shifts the random numbers of another block. |
| Data integrity | The raw files are the provenance anchor. A hash check proves that no file was edited, truncated or re-encoded. |
| Repository layout | Shows where every artefact lives and which governance files exist. |

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd()
ROOT = next(p for p in [_here, *_here.parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config as C, plots, results_io as R

import hashlib, importlib.metadata as md, platform
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 140, "display.max_colwidth", 90)
print("repository root (relative checks only):", ROOT.name)

repository root (relative checks only): Assignment


## Software environment

**Why.** The pinned versions in `requirements.txt` were resolved once; the table compares them with what is actually running. A mismatch is not necessarily an error, but it has to be visible.

In [2]:
PINNED = {}
for line in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        name, ver = line.split("==")
        PINNED[name.strip().lower()] = ver.strip()

PACKAGES = ["numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib",
            "seaborn", "pyarrow", "networkx", "PyYAML", "jupytext", "nbclient"]
rows = []
for pkg in PACKAGES:
    try:
        installed = md.version(pkg)
    except md.PackageNotFoundError:
        installed = "missing"
    pinned = PINNED.get(pkg.lower(), "-")
    rows.append({"package": pkg, "pinned": pinned, "installed": installed,
                 "status": "ok" if pinned in ("-", installed) else "differs"})
env = pd.DataFrame(rows)
display(env)
print(f"Python {platform.python_version()} on {platform.system()} {platform.release()}")

# The estimators and statistics used later must be importable: fail here, not in section 6.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import SplineTransformer
assert "missing" not in set(env["installed"]), env[env["installed"] == "missing"]
n_diff = int((env["status"] == "differs").sum())
print(f"{n_diff} package(s) differ from the pinned version" if n_diff else "all pinned versions match")

,package,pinned,installed,status
0,numpy,2.2.3,2.2.3,ok
1,pandas,2.2.3,2.2.3,ok
2,scipy,1.16.3,1.16.3,ok
3,scikit-learn,1.9.1,1.9.1,ok
4,statsmodels,0.15.0,0.15.0,ok
5,matplotlib,3.10.8,3.10.8,ok
6,seaborn,0.13.2,0.13.2,ok
7,pyarrow,26.0.0,26.0.0,ok
8,networkx,3.7,3.7,ok
9,PyYAML,6.0.2,6.0.2,ok


Python 3.12.5 on Windows 11


all pinned versions match


**So what.** Every package that later sections need is importable, and the table above records the exact versions behind the submitted numbers. Differences from the pins (if any) are listed, not hidden.

## Run mode and derived sizes

**Why.** Repeated cross-validation, bootstrap and permutation counts dominate run time. `NB_MODE` (environment variable, default `FAST`) switches all of them together from one place (`src/config.py`), so no section hard-codes a size.

In [3]:
modes = pd.DataFrame({
    "setting": ["CV splits x repeats", "bootstrap replicates", "coefficient bootstrap", "permutations", "robustness seeds"],
    "FAST": ["5 x 2", 100, 100, 20, 10],
    "FINAL": ["5 x 5", 1000, 500, 100, 10],
    f"active ({C.NB_MODE})": [f"{C.CV_SPLITS} x {C.CV_REPEATS}", C.N_BOOT, C.N_BOOT_COEF, C.N_PERM, C.N_ROBUSTNESS_SEEDS],
})
display(modes)
expected = {"FAST": (2, 100, 100, 20), "FINAL": (5, 1000, 500, 100)}[C.NB_MODE]
assert (C.CV_REPEATS, C.N_BOOT, C.N_BOOT_COEF, C.N_PERM) == expected, "config sizes do not match the documented mode"
print(f"NB_MODE = {C.NB_MODE}; holdout fraction {C.HOLDOUT_FRAC:.0%}; SESOI ROC-AUC {C.SESOI['roc_auc']}")

,setting,FAST,FINAL,active (FAST)
0,CV splits x repeats,5 x 2,5 x 5,5 x 2
1,bootstrap replicates,100,1000,100
2,coefficient bootstrap,100,500,100
3,permutations,20,100,20
4,robustness seeds,10,10,10


NB_MODE = FAST; holdout fraction 20%; SESOI ROC-AUC 0.01


**So what.** The active mode is printed with its derived sizes. The documented FAST and FINAL sizes agree with `config.py` (asserted), so the table cannot drift from the code.

## Seed derivation

**Why.** A single integer seed shared by every random step would couple them: adding one bootstrap call would change the cross-validation folds. `numpy.random.SeedSequence(SEED_MASTER).spawn(k)` produces statistically independent child streams from one master value. Names are bound to fixed positions, so a stream never moves when another is added at the end. Robustness checks (Stage S9) vary only the CV and bootstrap streams; the 20% hold-out is generated once and stored.

In [4]:
STREAMS = ["split", "cv", "bootstrap", "permutation", "robustness_base"]
root_ss = np.random.SeedSequence(C.SEED_MASTER)
children = root_ss.spawn(len(STREAMS))
seed_table = pd.DataFrame({
    "stream": STREAMS,
    "spawn_index": range(len(STREAMS)),
    "integer_seed": [int(c.generate_state(1)[0]) for c in children],
})
display(seed_table)

# Reproducibility: re-deriving gives identical draws; different streams give different draws.
again = np.random.SeedSequence(C.SEED_MASTER).spawn(len(STREAMS))
draws_a = [np.random.default_rng(c).random(3) for c in children]
draws_b = [np.random.default_rng(c).random(3) for c in again]
assert all(np.array_equal(a, b) for a, b in zip(draws_a, draws_b))
assert len({tuple(d) for d in draws_a}) == len(STREAMS)
robust_seeds = [int(s.generate_state(1)[0]) for s in children[-1].spawn(C.N_ROBUSTNESS_SEEDS)]
assert len(set(robust_seeds)) == C.N_ROBUSTNESS_SEEDS
print(f"SEED_MASTER = {C.SEED_MASTER}; {len(STREAMS)} named streams; {len(robust_seeds)} robustness seeds, first three {robust_seeds[:3]}")

,stream,spawn_index,integer_seed
0,split,0,1095125285
1,cv,1,975503932
2,bootstrap,2,3647788826
3,permutation,3,2081790035
4,robustness_base,4,3547934149


SEED_MASTER = 20261011; 5 named streams; 10 robustness seeds, first three [2612742270, 527626176, 2524347206]


**So what.** The master seed is `20261011`. Re-deriving it reproduces identical draws (asserted) and the five streams are mutually distinct. The ten robustness seeds are children of one stream, so they are reproducible without being typed in by hand.

## Data integrity

**Why.** The 16 NHANES files in `data/nhanes_2021_2023/raw/` are the unmodified CDC downloads (2026-10-10). `MANIFEST.sha256` stores one SHA-256 per file. Re-hashing at the start of the notebook proves that the analysis starts from the published data: a changed byte (a text editor re-saving a file, a line-ending conversion, a truncated download) would change the hash. `.gitattributes` marks `*.xpt` as binary for the same reason. The check is done twice, independently: here with `hashlib`, and (when available) through the project loader.

In [5]:
def read_manifest(path):
    entries = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        digest, name = line.split(None, 1)
        entries[name.strip().lstrip("*")] = digest.lower()
    return entries

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

manifest = read_manifest(C.MANIFEST)
rows = []
for stem in C.NHANES_FILES:
    fname = f"{stem}.xpt"
    p = C.RAW_DIR / fname
    actual = sha256_of(p) if p.exists() else None
    rows.append({
        "file": fname,
        "size_kb": round(p.stat().st_size / 1024, 1) if p.exists() else None,
        "codebook": (C.CODEBOOK_DIR / f"{stem}.htm").exists(),
        "sha256_manifest": manifest.get(fname, "")[:12] + "...",
        "sha256_actual": (actual or "")[:12] + "...",
        "match": actual is not None and actual == manifest.get(fname),
    })
integrity = pd.DataFrame(rows)
display(integrity)

assert set(manifest) == {f"{s}.xpt" for s in C.NHANES_FILES}, "manifest and file list differ"
assert integrity["match"].all(), integrity[~integrity["match"]]
assert integrity["codebook"].all(), "codebook missing for a raw file"
print(f"{int(integrity['match'].sum())}/{len(integrity)} files match MANIFEST.sha256; "
      f"total {integrity['size_kb'].sum() / 1024:.1f} MB; every file has a CDC codebook")

,file,size_kb,codebook,sha256_manifest,sha256_actual,match
0,DEMO_L.xpt,2521.6,True,ca4374a158b4...,ca4374a158b4...,True
1,GHB_L.xpt,169.9,True,67aee0353160...,67aee0353160...,True
2,GLU_L.xpt,126.2,True,70f773ab37e3...,70f773ab37e3...,True
3,BMX_L.xpt,1526.6,True,44440c416d9a...,44440c416d9a...,True
4,BPXO_L.xpt,680.4,True,189aa9bed689...,189aa9bed689...,True
5,TCHOL_L.xpt,253.4,True,5c3312e555bb...,5c3312e555bb...,True
6,HDL_L.xpt,253.4,True,c1bb43df5ca3...,c1bb43df5ca3...,True
7,TRIGLY_L.xpt,314.3,True,77679864c33b...,77679864c33b...,True
8,DIQ_L.xpt,827.7,True,9535a023673a...,9535a023673a...,True
9,BPQ_L.xpt,400.1,True,0acde3af0526...,0acde3af0526...,True


16/16 files match MANIFEST.sha256; total 12.2 MB; every file has a CDC codebook


In [6]:
# Second, independent route through the project loader (skipped until nhanes_io exists).
try:
    from src import nhanes_io
    status = nhanes_io.ensure_raw_data(download=False)
    print("nhanes_io.ensure_raw_data(download=False) ->", status if len(str(status)) < 300 else "ok")
except (ImportError, AttributeError) as exc:
    print("project loader not available yet, hashlib check above is the authority:", type(exc).__name__)

nhanes_io.ensure_raw_data(download=False) -> ok


**So what.** All 16 raw files are byte-identical to the manifest and each has a matching codebook, so provenance is intact: every later number can be traced to the CDC release of 2021-08 to 2023-08. If this cell fails, nothing downstream should be trusted.

## Repository layout and governance files

**Why.** The marking criteria reward a traceable process. The table maps each stage to its code, results and governance artefacts, and checks that the S0 artefacts exist and have the documented columns (process document section 7.2).

In [7]:
LAYOUT = pd.DataFrame([
    ("config/", "variable_spec.csv: single source of truth for variables", "S2"),
    ("src/", "importable modules (config, plots, results_io, nhanes_io, survey, labels, prep, evalkit, ...)", "S0-S9"),
    ("notebooks/sections/", "one jupytext percent script per section of the final notebook", "S0-S10"),
    ("notebooks/executed/", "executed section notebooks (tools/run_section.py)", "S0-S10"),
    ("governance/", "register fragments, hypotheses, claims, GenAI log, devlog, scope document", "S0-S10"),
    ("results/", "metrics_<S>.json per section and section tables", "S0-S10"),
    ("figures/", "<S>_<name>.png with captions", "S0-S10"),
    ("data/nhanes_2021_2023/", "raw XPT (read-only), codebooks, MANIFEST.sha256, data notes", "S2"),
    ("tools/", "run_section.py, check_repo.py, build_final.py", "S0, S11"),
], columns=["path", "content", "stage"])
LAYOUT["exists"] = [(ROOT / p).exists() for p in LAYOUT["path"]]
display(LAYOUT)
assert LAYOUT.loc[LAYOUT["path"].isin(["src/", "notebooks/sections/", "governance/", "data/nhanes_2021_2023/", "tools/"]), "exists"].all()

REGISTER_COLS = ["dr_id", "stage", "date", "type", "finding", "evidence_ref", "evidence_value", "decision",
                 "alternatives_rejected", "hyp_id", "status", "check_verdict", "check_note", "loop_ref",
                 "change_request", "commit"]
GENAI_COLS = ["date", "tool_version", "purpose", "data_shared", "output_used_in", "human_verification"]

gov = C.GOVERNANCE_DIR
checks = []
for rel, cols in [("register/S0.csv", REGISTER_COLS), ("genai_log.csv", GENAI_COLS)]:
    df = pd.read_csv(gov / rel)
    checks.append({"artefact": f"governance/{rel}", "rows": len(df), "columns_ok": list(df.columns) == cols})
for rel in ["README.md", "devlog.md"]:
    checks.append({"artefact": f"governance/{rel}", "rows": None, "columns_ok": (gov / rel).exists()})
gov_check = pd.DataFrame(checks)
display(gov_check)
assert gov_check["columns_ok"].all(), gov_check
reg0 = pd.read_csv(gov / "register/S0.csv")
assert len(reg0) >= 1 and (pd.read_csv(gov / "genai_log.csv").shape[0] >= 1)
print(f"S0 acceptance: register rows = {len(reg0)}, GenAI log rows = {pd.read_csv(gov / 'genai_log.csv').shape[0]}")

,path,content,stage,exists
0,config/,variable_spec.csv: single source of truth for variables,S2,True
1,src/,"importable modules (config, plots, results_io, nhanes_io, survey, labels, prep, evalki...",S0-S9,True
2,notebooks/sections/,one jupytext percent script per section of the final notebook,S0-S10,True
3,notebooks/executed/,executed section notebooks (tools/run_section.py),S0-S10,True
4,governance/,"register fragments, hypotheses, claims, GenAI log, devlog, scope document",S0-S10,True
5,results/,metrics_<S>.json per section and section tables,S0-S10,True
6,figures/,<S>_<name>.png with captions,S0-S10,True
7,data/nhanes_2021_2023/,"raw XPT (read-only), codebooks, MANIFEST.sha256, data notes",S2,True
8,tools/,"run_section.py, check_repo.py, build_final.py","S0, S11",True


,artefact,rows,columns_ok
0,governance/register/S0.csv,8.0,True
1,governance/genai_log.csv,6.0,True
2,governance/README.md,NaN,True
3,governance/devlog.md,NaN,True


S0 acceptance: register rows = 8, GenAI log rows = 6


**So what.** The governance artefacts required by Stage S0 exist and their headers are identical to the documented schema, so the per-stage fragments can be merged into one decision register without manual repair.

## Freeze order

**Why.** Results may only be *down-graded* after a freeze, never re-run until they look better. Fixing the order up front, with a git tag at each point, lets the commit history prove that hypotheses preceded the ablations.

In [8]:
FREEZES = pd.DataFrame([
    ("scope-freeze", "end of S3", "variable scope and availability rule"),
    ("target-freeze", "end of S5", "label L_comb, populations E-ALL / E-SCREEN, 3x3 strata"),
    ("prereg-lock", "end of S6", "hypotheses.md (SHA-256 + timestamp), before the first ablation commit"),
    ("exp-freeze", "end of S9", "experiments; hold-out evaluated once afterwards"),
    ("content-freeze", "end of S11", "notebook, slides and claims"),
    ("submitted-v1", "S12", "submission"),
], columns=["tag", "when", "what is fixed"])
display(FREEZES)

,tag,when,what is fixed
0,scope-freeze,end of S3,variable scope and availability rule
1,target-freeze,end of S5,"label L_comb, populations E-ALL / E-SCREEN, 3x3 strata"
2,prereg-lock,end of S6,"hypotheses.md (SHA-256 + timestamp), before the first ablation commit"
3,exp-freeze,end of S9,experiments; hold-out evaluated once afterwards
4,content-freeze,end of S11,"notebook, slides and claims"
5,submitted-v1,S12,submission


In [9]:
payload = {
    "mode": C.NB_MODE,
    "seed_master": C.SEED_MASTER,
    "streams": dict(zip(seed_table["stream"], seed_table["integer_seed"])),
    "robustness_seeds": robust_seeds,
    "n_raw_files_verified": int(integrity["match"].sum()),
    "raw_total_mb": round(float(integrity["size_kb"].sum() / 1024), 2),
    "n_package_differs": n_diff,
    "python": platform.python_version(),
    "packages": dict(zip(env["package"], env["installed"])),
}
R.save_section("S0", payload)
(ROOT / "results" / "S0").mkdir(exist_ok=True)
env.to_csv(ROOT / "results" / "S0" / "environment.csv", index=False)
print("saved results/metrics_S0.json")
print("S0 OK")

saved results/metrics_S0.json
S0 OK
